In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats

In [ ]:
plt.style.use("dark_background")
df = yf.download(
    "AVGO",
    start="2020-01-01",
    end="2025-02-01",
    auto_adjust=False
)

In [ ]:
df = df.sort_values("Date")

df = df.reset_index(drop=True) 
df.head()
#dataframe showing the first 5 rows of the dataset56

In [ ]:
df.info()

In [ ]:
df.index

In [ ]:
prices = df["Adj Close"]
prices.plot()

In [ ]:
prices.iloc[-1]

In [ ]:
prices.head()



In [ ]:
prices.isna().sum()

In [ ]:
prices.describe()

In [ ]:
prices / prices.shift(1)
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns = log_returns.dropna()
log_returns.describe()

Compare downside vs upside spread:
Downside (Median → Q1):   /50% → 25%
0.13 − (−1.16) ≈ 1.29%
Upside (Q3 → Median):     /75%→50%
1.42 − 0.13 ≈ 1.29%
Symmetrical → returns are fairly balanced
25% quartile (Q1) = −1.16%
50% quartile (Median) = +0.13%
75% quartile (Q3) = +1.42%
below x% of daily log returns was =
above 100-x% was above =
eg. 25% of all daily returns are worse than −1.16%
    75% are better than −1.16%

In [ ]:
log_returns.isna().sum()
#counts how many missing values exist in the entire series

In [ ]:
log_returns.plot()


In [ ]:
log_returns.hist(bins=50)

In [ ]:
mu_hat = log_returns.mean()

In [ ]:
sigma_hat = log_returns.std()


sigma(std):
Typical daily fluctuation size
Width of the return distribution

In [ ]:
mu_annual = mu_hat * 252
sigma_annual = sigma_hat * np.sqrt(252)

Var  = σ^2
VarT ​= T⋅σ^2
σT   ​= sqrt(T⋅σ^2)
σT   ​=σ.sqrt(T)

Why this works

Returns add linearly over time rmeanx252
→ Mean scales with time 
Variances add, not standard deviations 
→ Std scales with √time

This relies on:
Independence (or weak dependence)
Identically distributed returns

In [ ]:
print(mu_hat, sigma_hat)

In [ ]:
n_days = 252
n_paths = 10000

What could happen over the next year?”
10000 number of alternate futures

In [ ]:
sim_returns = np.random.normal(
    mu_hat,
    sigma_hat,
    size=(n_paths, n_days)
)


Path 1: r₁, r₂, r₃, ... r₂₅₂
Path 2: r₁, r₂, r₃, ... r₂₅₂
...
Path 10000

Each row → one possible future
Each column → one day forward in time
Each cell → one simulated daily return

Law of Large Numbers
Monte Carlo works because repeated sampling converges to the true distribution implied by the model

In [ ]:
cum_returns = np.cumsum(sim_returns, axis=1)

Each row is daily log returns

But prices depend on accumulated returns
axis=1 → sum horizontally

In [ ]:
S0 = prices.iloc[-1].values[0]
type(S0)
#Starting point for all futures simulations

Take the most recent price in the dataset and store it as the starting price (S₀) for the simulation.
iloc = integer location index
-1 → last element
-2 → second last

In [ ]:
sim_prices = S0 * np.exp(cum_returns)
#Log returns sum → cumulative log return
#Exponent converts log-space → price-space

In [ ]:
sim_prices[:5, :5]

In [ ]:
plt.plot(sim_prices[:50].T, alpha=0.2)
plt.xlabel("Days")
plt.ylabel("Price")
plt.show()

In [ ]:
final_prices = sim_prices[:, -1]
plt.hist(final_prices, bins=50)
plt.xlabel("Price in 1 year")
plt.ylabel("Frequency")
plt.show()
#right skewed distribution 

In [ ]:
final_prices.mean()

In [ ]:
VaR_5 = np.percentile(final_prices, 5)
VaR_5
#95% of simulated outcomes end above this price , 5% end below
#95% confidence, the price will not fall below VaR5 in one year
#left tail 

In [ ]:
VaR_loss = S0 - VaR_5
VaR_loss
#In the worst 5% of cases, the loss exceeds this amount

In [ ]:
ES_5 = final_prices[final_prices <= VaR_5].mean()
ES_5
#how bad the tail is
#EXPECTED SHORTFALL (ES) or Conditional Value at Risk (CVaR)
#5% ES =“Average outcome given that we are in the worst 5%.”

In [ ]:
plt.hist(final_prices, bins=50)
plt.axvline(VaR_5, color="red", linestyle="--", label="VaR 5%")
plt.axvline(ES_5, color="purple", linestyle="--", label="ES 5%")
plt.legend()
plt.show()